### Import modules

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout

# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


### Load Data

In [16]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/embeddings/Gemini_embeddings.csv')

In [11]:
data

,0,1,2,3,4,5,6,7,8,9,...,690,691,692,693,694,695,696,697,698,699
0,0.063873,0.029166,-0.092841,0.082908,-0.045293,0.048373,-0.024144,-0.098575,0.073264,0.015625,...,-0.000245,-0.000597,0.000415,-0.000052,0.000605,-0.000209,0.001199,0.000775,-0.001836,0.000389
1,0.299058,0.183732,0.041167,-0.060711,0.168758,0.008086,0.021418,-0.210063,-0.076003,0.076868,...,0.000952,-0.000337,0.001036,-0.000864,0.001016,-0.001975,-0.003087,-0.001347,0.000735,-0.001689
2,-0.118123,-0.004466,-0.099584,-0.018294,-0.025620,0.130191,-0.063593,-0.025870,0.049883,0.127452,...,0.000131,0.000065,0.001897,-0.002024,0.000321,-0.000626,-0.001224,0.000920,-0.000234,-0.001493
3,-0.142899,-0.027203,-0.077797,0.251772,-0.017832,0.126778,0.019879,0.012036,0.045680,0.161120,...,0.000643,0.000329,0.000760,0.001614,-0.000007,0.001234,-0.000897,0.000389,-0.001466,0.002405
4,-0.258978,-0.035706,0.074990,-0.032235,0.125542,0.034235,0.045474,-0.014249,0.010248,0.009401,...,-0.002477,-0.000314,0.001549,0.001559,-0.001205,-0.002138,-0.003368,-0.000270,-0.001675,-0.001454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1425,0.268183,0.125446,0.014547,0.021254,0.085115,0.012986,0.049597,-0.046481,-0.010605,-0.000143,...,0.000231,-0.000275,0.001251,-0.001090,-0.000520,-0.000811,-0.001114,-0.000451,-0.000505,-0.000945
1426,-0.025502,0.138609,-0.128277,-0.025648,-0.180123,-0.015693,-0.136549,-0.018124,-0.203595,-0.012750,...,-0.001094,0.000031,0.000843,-0.002569,0.000771,-0.000483,-0.001082,-0.000439,0.000792,0.001287
1427,-0.012895,0.022129,-0.297922,0.006917,-0.015824,0.021341,0.030012,-0.042817,-0.037995,0.123513,...,0.000317,0.000026,-0.000210,0.000150,-0.001195,-0.001283,0.000038,-0.000380,-0.000284,-0.000691
1428,0.301530,0.154765,0.007250,-0.066169,0.176450,-0.020861,0.003444,-0.144786,-0.094255,0.054136,...,-0.001418,0.000375,-0.002174,-0.000679,-0.001155,0.001610,0.000425,-0.000057,-0.000598,0.002748


In [13]:
from sklearn.decomposition import PCA
pca = PCA(n_components=700)
dx = pca.fit_transform(data.drop(columns=['label']))
lc = data["label"]
data = pd.DataFrame(dx)
data['label'] = lc


### NN Architecture

In [17]:
TARGET_COLUMN = "label"

df = data
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_features:
    raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

X = df[numeric_features].fillna(0.0).values
y = df[TARGET_COLUMN].astype(str).values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)
X_val, X_test, y_val_enc, y_test_enc = train_test_split(
    X_temp,
    y_temp_enc,
    test_size=0.5,
    random_state=42,
    stratify=y_temp_enc,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)

def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            Dense(num_classes, activation="softmax"),
        ]
    )

model = build_classifier(X_train.shape[1], num_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=96,
    callbacks=callbacks,
    verbose=1,
)

test_loss, test_acc, test_top3 = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

y_pred = model.predict(X_test)
y_pred_labels = y_pred.argmax(axis=1)
report = classification_report(
    y_test_enc,
    y_pred_labels,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
class_breakdown = (
    report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
)

print("\nKey metrics:")
display(summary)
print("\nTop classes by support:")
display(class_breakdown)

Epoch 1/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - accuracy: 0.2858 - loss: 2.7612 - top3_acc: 0.4476 - val_accuracy: 0.5105 - val_loss: 2.0126 - val_top3_acc: 0.7133 - learning_rate: 0.0010
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - accuracy: 0.2858 - loss: 2.7612 - top3_acc: 0.4476 - val_accuracy: 0.5105 - val_loss: 2.0126 - val_top3_acc: 0.7133 - learning_rate: 0.0010
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.6005 - loss: 1.5611 - top3_acc: 0.7928 - val_accuracy: 0.6294 - val_loss: 1.5136 - val_top3_acc: 0.7762 - learning_rate: 0.0010
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.6005 - loss: 1.5611 - top3_acc: 0.7928 - val_accuracy: 0.6294 - val_loss: 1.5136 - val_top3_acc: 0.7762 - learning_rate: 0.0010
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.7124 - loss: 1.1295 - top3_acc: 0.8899 - val_accuracy: 0.6713 - val_loss: 1.2930 - val_top3_acc: 0.8462 - learning_rate: 0.0010
Epoch 4/200
12/12 ━━━━━━━━━━━━

,precision,recall,f1-score,support
accuracy,0.846,0.846,0.846,0.846
macro avg,0.856,0.835,0.831,143.000
weighted avg,0.868,0.846,0.845,143.000



Top classes by support:


,precision,recall,f1-score,support
LLM Results Evaluation,0.889,0.800,0.842,10.0
Explainable AI (XAI) Techniques,1.000,1.000,1.000,10.0
Tool Use for LLMs,0.857,0.600,0.706,10.0
Structured Output & Formatting for LLMs,1.000,1.000,1.000,10.0
LLM KV Cache Optimization,1.000,0.900,0.947,10.0
Modular LLM Agent Architectures,0.667,1.000,0.800,10.0
LLMs for Recommender Systems,0.909,1.000,0.952,10.0
"LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",0.625,0.556,0.588,9.0


In [18]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 768)            │       590,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 768)            │         3,072 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 18)             │         2,322 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,464,888 (13.22 MB)

 Trainable params: 1,153,938 (4.40 MB)

 Non-trainable params: 3,072 (12.00 KB)

 Optimizer params: 2,307,878 (8.80 MB)

### Logistic Regression classifier

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

TARGET_COLUMN = "label"
feature_columns = [col for col in data.columns if col != TARGET_COLUMN]
X = data[feature_columns].values
y = data[TARGET_COLUMN].astype(str).values

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

logreg_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logreg",
            LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=2000,
                n_jobs=None,
                verbose=0,
            ),
        ),
    ]
)

logreg_pipeline.fit(X_train_lr, y_train_lr)
y_pred_lr = logreg_pipeline.predict(X_test_lr)
y_proba_lr = logreg_pipeline.predict_proba(X_test_lr)

logreg_accuracy = (y_pred_lr == y_test_lr).mean()
classes = logreg_pipeline.named_steps["logreg"].classes_
top3_indices = np.argsort(y_proba_lr, axis=1)[:, -3:]
top3_predictions = classes[top3_indices]
top3_accuracy = (top3_predictions == y_test_lr.reshape(-1, 1)).any(axis=1).mean()

print(f"LogReg test accuracy: {logreg_accuracy:.4f} | Top-3 accuracy: {top3_accuracy:.4f}")

logreg_report = pd.DataFrame(
    classification_report(y_test_lr, y_pred_lr, target_names=classes, output_dict=True, zero_division=0)
).T

summary_metrics = logreg_report.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
per_class_metrics = (
    logreg_report.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).round(3)
)

print("\nLogistic Regression summary metrics:")
display(summary_metrics)
print("\nPer-class breakdown:")
display(per_class_metrics)


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg test accuracy: 0.8147 | Top-3 accuracy: 0.9266

Logistic Regression summary metrics:


,precision,recall,f1-score,support
accuracy,0.815,0.815,0.815,0.815
macro avg,0.809,0.805,0.805,286.000
weighted avg,0.817,0.815,0.813,286.000



Per-class breakdown:


,precision,recall,f1-score,support
LLM Results Evaluation,0.875,0.700,0.778,20.0
LLM KV Cache Optimization,0.952,1.000,0.976,20.0
Tool Use for LLMs,0.579,0.550,0.564,20.0
Modular LLM Agent Architectures,0.800,0.800,0.800,20.0
LLMs for Recommender Systems,0.864,0.950,0.905,20.0
"Reliable, Transparent, & Augmented LLMs",0.571,0.632,0.600,19.0
Structured Output & Formatting for LLMs,1.000,0.947,0.973,19.0
LLM Agent Training & Alignment,0.895,0.895,0.895,19.0
Enhanced User Intent Comprehension with LLMs,0.826,1.000,0.905,19.0
Explainable AI (XAI) Techniques,1.000,1.000,1.000,19.0
